#PAGOS POR LEY 

In [1]:
import pandas as pd
import pdfplumber
import re
import os
from glob import glob

def extract_dl25844_by_context_with_period(pdf_path, periodo_interrupcion):
    """
    Método que analiza el contexto alrededor de "25844" para identificar montos correctos
    y verifica que coincida con el periodo de interrupción
    """
    try:
        # Formatear periodo de interrupción para coincidir con el formato del PDF (YYYY-MM)
        periodo_buscar = f"{periodo_interrupcion[:4]}-{periodo_interrupcion[4:6]}"
        print(f" 🔍 Buscando DL 25844 para periodo: {periodo_buscar}")
        
        with pdfplumber.open(pdf_path) as pdf:
            dl_amounts = []

            for page in pdf.pages:
                text = page.extract_text()
                lines = text.split('\n')

                # Buscar la sección de "DETALLE DE IMPORTES" o similar
                in_detalle_section = False

                for i, line in enumerate(lines):
                    line_clean = line.strip()

                    # Marcar cuando estamos en la sección de detalles
                    if 'DETALLE DE IMPORTES' in line_clean or 'TOTAL Mes Actual' in line_clean:
                        in_detalle_section = True
                        continue

                    # Buscar líneas con 25844 solo en la sección de detalles
                    if in_detalle_section and '25844' in line_clean:
                        print(f" 📄 En sección detalles: {line_clean}")

                        # Excluir conceptos no relacionados
                        if any(exclude in line_clean for exclude in ['Redondeo', 'Mora', 'Interés']):
                            print(f" ⚠️ Excluido (concepto no relacionado)")
                            continue

                        # VERIFICAR PERIODO: Solo procesar si coincide con el periodo de interrupción
                        if periodo_buscar in line_clean:
                            print(f" ✅ Coincide periodo {periodo_buscar}")
                            
                            # Buscar montos específicos de DL 25844
                            if 'Cal' in line_clean or 'Sum' in line_clean:
                                # Buscar número negativo en esta línea
                                negative_matches = re.findall(r'(-\d+\.\d{2})', line_clean)
                                for match in negative_matches:
                                    amount = float(match)
                                    # CONVERTIR A POSITIVO
                                    amount_positive = abs(amount)
                                    dl_amounts.append(amount_positive)
                                    print(f" ✅ Monto DL 25844 encontrado: {amount} -> Convertido a positivo: {amount_positive}")
                        else:
                            print(f" ⚠️ Periodo no coincide: esperado {periodo_buscar}")

            if dl_amounts:
                # Formatear montos como string separados por comas (ya en positivo)
                montos_formateados = ", ".join([f"{amount:.2f}" for amount in dl_amounts])
                print(f" 🧮 MONTOS DL 25844 ENCONTRADOS: {montos_formateados}")
                return montos_formateados, len(dl_amounts)
            else:
                return None, 0

    except Exception as e:
        print(f" ✗ Error en extracción por contexto: {e}")
        return None, 0

def extract_dl25844_detailed_sum_with_period(pdf_path, periodo_interrupcion):
    """
    Método más preciso que busca SOLO montos DL 25844 con verificación de periodo
    """
    try:
        periodo_buscar = f"{periodo_interrupcion[:4]}-{periodo_interrupcion[4:6]}"
        print(f" 🔍 Buscando DL 25844 específicos para periodo: {periodo_buscar}")
        
        with pdfplumber.open(pdf_path) as pdf:
            all_dl_amounts = []

            for page_num, page in enumerate(pdf.pages):
                text = page.extract_text()
                lines = text.split('\n')

                for i, line in enumerate(lines):
                    line_clean = line.strip()

                    # FILTRO MÁS ESTRICTO: debe contener 25844 y estar relacionado con Cal/Sum
                    if '25844' in line_clean:
                        # Excluir líneas que contengan "Redondeo" u otros conceptos no relacionados
                        if any(exclude in line_clean for exclude in ['Redondeo', 'Mora', 'Interés']):
                            continue

                        # VERIFICAR PERIODO
                        if periodo_buscar in line_clean:
                            print(f" 📍 Línea con 25844 y periodo {periodo_buscar}: {line_clean}")

                            # Buscar específicamente el patrón DL 25844 Cal_Sum AAAA-MM -X.XX
                            specific_patterns = [
                                r'DL\s*25844\s*Cal[_\s\.]?Sum\s*\d{4}-\d{2}\s*(-\d+\.\d{2})',
                                r'Cal[_\s\.]?Sum\s*\d{4}-\d{2}.*?25844.*?(-\d+\.\d{2})',
                            ]

                            for pattern in specific_patterns:
                                matches = re.findall(pattern, line_clean, re.IGNORECASE)
                                for match in matches:
                                    amount = float(match)
                                    # CONVERTIR A POSITIVO
                                    amount_positive = abs(amount)
                                    all_dl_amounts.append(amount_positive)
                                    print(f" ✅ Monto DL 25844 específico: {amount} -> Convertido a positivo: {amount_positive}")
                        else:
                            print(f" ⚠️ Periodo no coincide: {line_clean}")

            # Formatear montos como string separados por comas (ya en positivo)
            if all_dl_amounts:
                montos_formateados = ", ".join([f"{amount:.2f}" for amount in all_dl_amounts])
                print(f" 🧮 MONTOS DL 25844 ENCONTRADOS: {montos_formateados}")
                return montos_formateados, len(all_dl_amounts)
            else:
                return None, 0

    except Exception as e:
        print(f" ✗ Error en extracción detallada: {e}")
        return None, 0

def extract_all_dl25844_amounts_with_period(pdf_path, periodo_interrupcion):
    """
    Extrae SOLO los montos del descuento DL 25844 con verificación de periodo
    """
    try:
        periodo_buscar = f"{periodo_interrupcion[:4]}-{periodo_interrupcion[4:6]}"
        
        with pdfplumber.open(pdf_path) as pdf:
            all_amounts = []

            for page in pdf.pages:
                text = page.extract_text()
                if not text:
                    continue

                print(f" 🔍 Buscando DL 25844 específicos para periodo {periodo_buscar}...")

                # Buscar líneas que específicamente contengan DL 25844
                lines = text.split('\n')

                for i, line in enumerate(lines):
                    line_clean = line.strip()

                    # Solo procesar líneas que contengan "25844" Y ("Cal" o "Sum") Y el periodo correcto
                    if '25844' in line_clean and ('Cal' in line_clean or 'Sum' in line_clean) and periodo_buscar in line_clean:
                        print(f" 📄 Línea DL 25844 con periodo {periodo_buscar}: {line_clean}")

                        # Patrón MÁS ESPECÍFICO: buscar solo montos que estén en la misma línea de DL 25844
                        # y que no sean de "Redondeo"
                        if 'Redondeo' not in line_clean:
                            # Buscar números negativos específicamente asociados a DL 25844
                            dl_patterns = [
                                r'DL\s*25844\s*Cal[_\s\.]?Sum\s*\d{4}-\d{2}\s*(-\d+\.\d{2})',
                                r'25844.*?Cal[_\s\.]?Sum.*?(-\d+\.\d{2})',
                                r'DL\s*25844.*?(-\d+\.\d{2})',
                            ]

                            for pattern in dl_patterns:
                                matches = re.findall(pattern, line_clean, re.IGNORECASE)
                                for match in matches:
                                    try:
                                        amount = float(match)
                                        # CONVERTIR A POSITIVO
                                        amount_positive = abs(amount)
                                        all_amounts.append(amount_positive)
                                        print(f" ✅ Monto DL 25844 específico: {amount} -> Convertido a positivo: {amount_positive}")
                                    except ValueError:
                                        continue

            # Formatear montos como string separados por comas (ya en positivo)
            if all_amounts:
                montos_formateados = ", ".join([f"{amount:.2f}" for amount in all_amounts])
                print(f" 🧮 MONTOS DL 25844 ENCONTRADOS: {montos_formateados}")
                return montos_formateados, len(all_amounts)
            else:
                print(f" ❌ No se encontraron montos DL 25844 específicos para periodo {periodo_buscar}")
                return None, 0

    except Exception as e:
        print(f" ✗ Error procesando PDF: {e}")
        return None, 0

def debug_dl25844_search_with_period(pdf_path, periodo_interrupcion):
    """
    Función de debug mejorada para identificar correctamente DL 25844 con periodo
    """
    try:
        periodo_buscar = f"{periodo_interrupcion[:4]}-{periodo_interrupcion[4:6]}"
        
        with pdfplumber.open(pdf_path) as pdf:
            print(f"\n 🐛 DEBUG DETALLADO para {os.path.basename(pdf_path)} - Periodo: {periodo_buscar}")

            for page_num, page in enumerate(pdf.pages):
                text = page.extract_text()
                lines = text.split('\n')

                print(f" 📄 Página {page_num + 1}:")
                print(" " + "=" * 50)

                for i, line in enumerate(lines):
                    if '25844' in line:
                        line_clean = line.strip()
                        
                        # Verificar si coincide con el periodo
                        periodo_coincide = periodo_buscar in line_clean
                        periodo_status = "✅" if periodo_coincide else "❌"
                        
                        print(f" {periodo_status} Línea {i}: {line_clean}")

                        # Clasificar el tipo de línea
                        if 'Redondeo' in line_clean:
                            print(f" ⚠️ ES REDONDEO - IGNORAR")
                        elif 'Cal' in line_clean or 'Sum' in line_clean:
                            if periodo_coincide:
                                print(f" ✅ ES DL 25844 VÁLIDO (periodo coincide)")
                                # Mostrar montos
                                numbers = re.findall(r'-?\d+\.\d{2}', line_clean)
                                if numbers:
                                    # Mostrar conversión a positivo
                                    numbers_positive = [f"{abs(float(num)):.2f}" for num in numbers]
                                    print(f" Montos originales: {numbers}")
                                    print(f" Montos positivos: {numbers_positive}")
                            else:
                                print(f" ⚠️ DL 25844 PERIODO NO COINCIDE")
                        else:
                            print(f" ❓ TIPO DESCONOCIDO")

                        print(" " + "=" * 50)

            return True

    except Exception as e:
        print(f" ✗ Error en debug: {e}")
        return False

def find_pdf_file(suministro, periodo_facturado, pdf_folder):
    """
    Encuentra el archivo PDF correspondiente
    """
    periodo_formatted = f"{periodo_facturado[:4]}_{periodo_facturado[4:6]}"

    patterns = [
        f"*{suministro}*{periodo_facturado}*.pdf",
        f"*{suministro}*{periodo_formatted}*.pdf",
        f"*{suministro}*.pdf",
        f"*{periodo_facturado}*{suministro}*.pdf",
    ]

    for pattern in patterns:
        matches = glob(os.path.join(pdf_folder, pattern))
        if matches:
            print(f" 📁 PDF encontrado: {os.path.basename(matches[0])}")
            return matches[0]

    return None

def main():
    # Configurar rutas específicas
    excel_path = r"C:\Users\P760033411\OneDrive - Pluz Energía Perú S.A.A\EJECUCION--PYTHON\MuestraEDN.xlsx"
    pdf_folder = r"C:\Users\P760033411\OneDrive - Pluz Energía Perú S.A.A\EJECUCION--PYTHON\Recibos 1\Recibos"
    output_folder = r"C:\Users\P760033411\OneDrive - Pluz Energía Perú S.A.A\EJECUCION--PYTHON"

    # Verificar rutas
    if not os.path.exists(excel_path):
        print(f"❌ Error: No se encuentra el archivo Excel en {excel_path}")
        return

    if not os.path.exists(pdf_folder):
        print(f"❌ Error: No se encuentra la carpeta de PDFs en {pdf_folder}")
        return

    os.makedirs(output_folder, exist_ok=True)

    # Leer Excel
    print("📖 Leyendo archivo Excel...")
    df = pd.read_excel(excel_path)
    comp_lce_values = []
    count_values = []  # Nueva lista para contar valores por periodo

    print(f"🔍 Iniciando procesamiento de {len(df)} registros...")
    print("=" * 60)

    for index, row in df.iterrows():
        suministro = str(row['Suministro'])
        periodo_facturado = str(row['Periodo Facturado'])
        periodo_interrupcion = str(row['Periodo de la Interrupción'])
        item = row['Ítem']

        print(f"\n🔹 Ítem {item}: Suministro {suministro}, Periodo Facturado {periodo_facturado}, Periodo Interrupción {periodo_interrupcion}")

        # Buscar archivo PDF
        pdf_path = find_pdf_file(suministro, periodo_facturado, pdf_folder)

        if pdf_path and os.path.exists(pdf_path):
            # Primero hacer debug para identificar correctamente
            debug_dl25844_search_with_period(pdf_path, periodo_interrupcion)

            # Intentar diferentes métodos de extracción CON PERIODO
            montos_formateados = None
            cantidad_montos = 0

            # Método 1: Por contexto (más preciso)
            montos_formateados, cantidad_montos = extract_dl25844_by_context_with_period(pdf_path, periodo_interrupcion)

            # Método 2: Detallado
            if montos_formateados is None:
                montos_formateados, cantidad_montos = extract_dl25844_detailed_sum_with_period(pdf_path, periodo_interrupcion)

            # Método 3: General
            if montos_formateados is None:
                montos_formateados, cantidad_montos = extract_all_dl25844_amounts_with_period(pdf_path, periodo_interrupcion)

            comp_lce_values.append(montos_formateados)
            count_values.append(cantidad_montos)
        else:
            print(f" ❌ No se encontró PDF correspondiente")
            comp_lce_values.append(None)
            count_values.append(0)

    # Actualizar DataFrame con las nuevas columnas
    df['Comp.LCE'] = comp_lce_values
    df['Cantidad_Compensaciones'] = count_values  # Nueva columna
    
    output_path = os.path.join(output_folder, "MuestraEDN_2020_110Casos_CompLCE_FINAL.xlsx")
    df.to_excel(output_path, index=False)

    print("\n" + "=" * 60)
    print("✅ PROCESAMIENTO COMPLETADO!")
    print(f"💾 Archivo guardado en: {output_path}")

    # Estadísticas
    encontrados = df['Comp.LCE'].notna().sum()
    total_montos = sum(count_values)
    
    print(f"\n📊 ESTADÍSTICAS FINALES:")
    print(f" Total de registros procesados: {len(df)}")
    print(f" Registros con compensaciones encontradas: {encontrados}")
    print(f" Total de montos individuales encontrados: {total_montos}")
    print(f" Tasa de éxito: {(encontrados/len(df))*100:.1f}%")
    if encontrados > 0:
        print(f" Promedio de montos por registro: {total_montos/encontrados:.1f}")

    # Mostrar distribución de cantidades
    print(f"\n📈 DISTRIBUCIÓN DE COMPENSACIONES POR REGISTRO:")
    for i in range(1, max(count_values) + 1):
        count = count_values.count(i)
        if count > 0:
            print(f"  {i} compensación(es): {count} registro(s)")

    # Mostrar resultados
    if encontrados > 0:
        print(f"\n🔢 MONTOS DL 25844 ENCONTRADOS (primeros 10 registros):")
        resultados = df[df['Comp.LCE'].notna()][['Ítem', 'Suministro', 'Periodo de la Interrupción', 'Comp.LCE', 'Cantidad_Compensaciones']].head(10)
        for _, row in resultados.iterrows():
            print(f" Ítem {row['Ítem']} - Suministro {row['Suministro']}")
            print(f"   Periodo: {row['Periodo de la Interrupción']}")
            print(f"   Montos: {row['Comp.LCE']}")
            print(f"   Cantidad: {row['Cantidad_Compensaciones']} compensación(es)")
            print()

if __name__ == "__main__":
    main()

📖 Leyendo archivo Excel...
🔍 Iniciando procesamiento de 149 registros...

🔹 Ítem 1: Suministro 236994, Periodo Facturado 202510, Periodo Interrupción 202510
 📁 PDF encontrado: RECIBOS_2025_10_236994.pdf

 🐛 DEBUG DETALLADO para RECIBOS_2025_10_236994.pdf - Periodo: 2025-10
 📄 Página 1:
 📄 Página 2:
 🔍 Buscando DL 25844 para periodo: 2025-10
 🔍 Buscando DL 25844 específicos para periodo: 2025-10
 🔍 Buscando DL 25844 específicos para periodo 2025-10...
 🔍 Buscando DL 25844 específicos para periodo 2025-10...
 ❌ No se encontraron montos DL 25844 específicos para periodo 2025-10

🔹 Ítem 2: Suministro 504180, Periodo Facturado 202509, Periodo Interrupción 202509
 📁 PDF encontrado: RECIBOS_2025_09_504180.pdf

 🐛 DEBUG DETALLADO para RECIBOS_2025_09_504180.pdf - Periodo: 2025-09
 📄 Página 1:
 ❌ Línea 38: FISE MENSAJE AL CLIENTE DL 25844 Cal_Sum 2025-07 -28.37
 ⚠️ DL 25844 PERIODO NO COINCIDE
 📄 Página 2:
 🔍 Buscando DL 25844 para periodo: 2025-09
 📄 En sección detalles: FISE MENSAJE AL CLIENT

**PAGO POR NORMA**

In [3]:
import pandas as pd
import pdfplumber
import re
import os
from glob import glob

def find_pdf_file(suministro, pdf_folder):
    """
    Encuentra el archivo PDF correspondiente
    """
    print(f" 🔍 Buscando PDF para suministro: {suministro}")
    
    # Buscar el PDF por el código de suministro
    patterns = [
        f"*{suministro}*.pdf",
        f"*{suministro}*.PDF",
        f"*{suministro.zfill(10)}*.pdf",  # Con ceros a la izquierda si es necesario
    ]
    
    for pattern in patterns:
        matches = glob(os.path.join(pdf_folder, pattern))
        if matches:
            print(f" 📁 PDF encontrado: {os.path.basename(matches[0])}")
            return matches[0]
    
    return None

def extract_ds026_from_pdf(pdf_path):
    """
    Extrae TODAS las compensaciones DS 026 del PDF
    Devuelve: lista de tuplas (mes_interrupcion, monto_compensado)
    """
    try:
        print(f"\n 📖 Extrayendo datos DS 026 del PDF: {os.path.basename(pdf_path)}")
        
        compensaciones = []
        
        with pdfplumber.open(pdf_path) as pdf:
            for page_num, page in enumerate(pdf.pages):
                # Extraer texto de la página
                text = page.extract_text()
                if not text:
                    continue
                
                # Dividir en líneas
                lines = text.split('\n')
                
                for line_num, line in enumerate(lines):
                    line_clean = line.strip()
                    
                    # Buscar líneas con DS 026 Cal_RCarga
                    if 'DS' in line_clean and '026' in line_clean and 'Cal_RCarga' in line_clean:
                        print(f" 📄 Encontrado en página {page_num+1}, línea {line_num}: {line_clean}")
                        
                        # Buscar patrón específico: DS 026 Cal_RCarga YYYY-MM -X.XX
                        # Ejemplo: "DS 026 Cal_RCarga 2025-06 -0.08"
                        
                        # Patrón para extraer mes y monto
                        patron = r'DS\s*0?26\s*Cal_RCarga\s*(\d{4}-\d{2})\s*(-?\d+\.\d{2})'
                        match = re.search(patron, line_clean)
                        
                        if match:
                            mes_interrupcion = match.group(1)  # Ej: "2025-06"
                            monto_str = match.group(2)        # Ej: "-0.08"
                            
                            try:
                                # Convertir monto a float y tomar valor absoluto
                                monto = float(monto_str)
                                monto_absoluto = abs(monto)
                                
                                # Convertir mes a formato YYYYMM (sin guion)
                                mes_formateado = mes_interrupcion.replace('-', '')
                                
                                # Agregar a la lista
                                compensaciones.append((mes_formateado, monto_absoluto))
                                print(f" ✅ Extraído: Mes={mes_formateado}, Monto={monto_absoluto:.2f}")
                                
                            except ValueError as e:
                                print(f" ⚠️ Error convirtiendo monto '{monto_str}': {e}")
                                continue
        
        print(f" 📊 Total de compensaciones DS 026 encontradas: {len(compensaciones)}")
        return compensaciones
            
    except Exception as e:
        print(f" ✗ Error procesando PDF: {e}")
        return []

def process_all_ds026_in_pdf(pdf_path):
    """
    Función para debug: muestra TODAS las ocurrencias de DS 026 en el PDF
    """
    print(f"\n{'='*60}")
    print(f" 🔎 ANÁLISIS COMPLETO DEL PDF")
    print(f" Archivo: {os.path.basename(pdf_path)}")
    print(f"{'='*60}")
    
    try:
        with pdfplumber.open(pdf_path) as pdf:
            for page_num, page in enumerate(pdf.pages):
                text = page.extract_text()
                if not text:
                    continue
                
                lines = text.split('\n')
                
                print(f"\n 📄 PÁGINA {page_num + 1}:")
                for line_num, line in enumerate(lines):
                    line_clean = line.strip()
                    
                    # Mostrar todas las líneas que contengan DS
                    if 'DS' in line_clean:
                        print(f"   Línea {line_num}: {line_clean}")
                        
                        # Intentar extraer datos si es DS 026
                        if '026' in line_clean:
                            patron = r'DS\s*0?26\s*Cal_RCarga\s*(\d{4}-\d{2})\s*(-?\d+\.\d{2})'
                            match = re.search(patron, line_clean)
                            if match:
                                print(f"     ✅ Coincidencia encontrada: Mes={match.group(1)}, Monto={match.group(2)}")
    
    except Exception as e:
        print(f" ✗ Error en análisis: {e}")

def main():
    # Configurar rutas específicas
    excel_path = r"C:\Users\P760033411\OneDrive - Pluz Energía Perú S.A.A\EJECUCION--PYTHON\EXTRAER-NORMA.xlsx"
    pdf_folder = r"C:\Users\P760033411\OneDrive - Pluz Energía Perú S.A.A\EJECUCION--PYTHON\Recibos 1\Recibos"
    output_folder = r"C:\Users\P760033411\OneDrive - Pluz Energía Perú S.A.A\EJECUCION--PYTHON"
    
    # Verificar rutas
    if not os.path.exists(excel_path):
        print(f"❌ Error: No se encuentra el archivo Excel en {excel_path}")
        return
    
    if not os.path.exists(pdf_folder):
        print(f"❌ Error: No se encuentra la carpeta de PDFs en {pdf_folder}")
        return
    
    os.makedirs(output_folder, exist_ok=True)
    
    # Leer Excel
    print("📖 Leyendo archivo Excel...")
    try:
        df = pd.read_excel(excel_path)
        print(f" ✅ Excel cargado. Filas: {len(df)}, Columnas: {list(df.columns)}")
        
        # Mostrar estructura del DataFrame
        print(f"\n 📋 Primeras 5 filas del Excel:")
        print(df.head())
        
    except Exception as e:
        print(f" ❌ Error cargando Excel: {e}")
        return
    
    # Verificar columnas requeridas
    required_columns = ['Código de Suministro']
    missing_columns = [col for col in required_columns if col not in df.columns]
    
    if missing_columns:
        print(f"❌ Error: Faltan columnas requeridas: {missing_columns}")
        print(f"   Columnas disponibles: {list(df.columns)}")
        return
    
    # Crear nuevas columnas para los resultados
    df['Mes_Interrupcion_PDF'] = None  # Mes extraído del PDF (formato YYYYMM)
    df['MONTO_PDF'] = None  # Monto compensado extraído del PDF
    df['Cantidad_Compensaciones'] = 0  # Número de compensaciones encontradas
    
    print(f"\n{'='*60}")
    print(f"🚀 INICIANDO PROCESAMIENTO DE {len(df)} REGISTROS")
    print(f"{'='*60}")
    
    # Diccionario para cachear resultados por suministro
    cache_suministros = {}
    
    # Procesar cada registro
    for index, row in df.iterrows():
        print(f"\n{'='*60}")
        print(f"📝 PROCESANDO REGISTRO {index + 1}/{len(df)}")
        
        try:
            suministro = str(row['Código de Suministro']).strip()
            print(f"   Suministro: {suministro}")
            
            # Buscar PDF
            pdf_path = find_pdf_file(suministro, pdf_folder)
            
            if pdf_path and os.path.exists(pdf_path):
                # Verificar si ya procesamos este suministro
                if suministro in cache_suministros:
                    print(f"   ✅ Usando datos en caché para suministro {suministro}")
                    compensaciones = cache_suministros[suministro]
                else:
                    # Procesar el PDF y extraer compensaciones
                    print(f"   🔍 Procesando PDF: {os.path.basename(pdf_path)}")
                    
                    # Opcional: hacer análisis completo del PDF
                    process_all_ds026_in_pdf(pdf_path)
                    
                    # Extraer compensaciones
                    compensaciones = extract_ds026_from_pdf(pdf_path)
                    
                    # Guardar en caché
                    cache_suministros[suministro] = compensaciones
                
                # Procesar compensaciones encontradas
                if compensaciones:
                    print(f"   📊 Compensaciones encontradas: {len(compensaciones)}")
                    
                    # Si hay múltiples compensaciones, las juntamos
                    meses = []
                    montos = []
                    
                    for mes, monto in compensaciones:
                        meses.append(mes)
                        montos.append(f"{monto:.2f}")
                    
                    # Asignar al DataFrame
                    df.at[index, 'Mes_Interrupcion_PDF'] = ", ".join(meses)
                    df.at[index, 'MONTO_PDF'] = ", ".join(montos)
                    df.at[index, 'Cantidad_Compensaciones'] = len(compensaciones)
                    
                    print(f"   ✅ Resultado: Mes(es)={df.at[index, 'Mes_Interrupcion_PDF']}")
                    print(f"                 Monto(s)={df.at[index, 'MONTO_PDF']}")
                    print(f"                 Cantidad={df.at[index, 'Cantidad_Compensaciones']}")
                else:
                    print(f"   ❌ No se encontraron compensaciones DS 026 en el PDF")
                    df.at[index, 'Mes_Interrupcion_PDF'] = "NO ENCONTRADO"
                    df.at[index, 'MONTO_PDF'] = "NO ENCONTRADO"
                    df.at[index, 'Cantidad_Compensaciones'] = 0
            else:
                print(f"   ❌ No se encontró PDF para el suministro {suministro}")
                df.at[index, 'Mes_Interrupcion_PDF'] = "PDF NO ENCONTRADO"
                df.at[index, 'MONTO_PDF'] = "PDF NO ENCONTRADO"
                df.at[index, 'Cantidad_Compensaciones'] = 0
                
        except Exception as e:
            print(f"   ❌ Error procesando registro {index}: {e}")
            df.at[index, 'Mes_Interrupcion_PDF'] = "ERROR"
            df.at[index, 'MONTO_PDF'] = "ERROR"
            df.at[index, 'Cantidad_Compensaciones'] = 0
    
    # Guardar resultados
    output_path = os.path.join(output_folder, "Resultados_DS026_Detallado.xlsx")
    df.to_excel(output_path, index=False)
    
    print(f"\n{'='*60}")
    print("✅ PROCESAMIENTO COMPLETADO!")
    print(f"💾 Resultados guardados en: {output_path}")
    
    # Mostrar resumen
    encontrados = sum(df['Cantidad_Compensaciones'] > 0)
    total_compensaciones = sum(df['Cantidad_Compensaciones'])
    
    print(f"\n📊 RESUMEN FINAL:")
    print(f"   Total registros procesados: {len(df)}")
    print(f"   Registros con compensaciones encontradas: {encontrados}")
    print(f"   Total de compensaciones individuales: {total_compensaciones}")
    print(f"   Suministros diferentes procesados: {len(cache_suministros)}")
    
    # Mostrar algunos resultados exitosos
    if encontrados > 0:
        print(f"\n🔢 EJEMPLOS DE COMPENSACIONES ENCONTRADAS:")
        resultados_exitosos = df[df['Cantidad_Compensaciones'] > 0][['Código de Suministro', 'Mes_Interrupcion_PDF', 'MONTO_PDF', 'Cantidad_Compensaciones']].head(10)
        
        for _, row in resultados_exitosos.iterrows():
            print(f"   Suministro: {row['Código de Suministro']}")
            print(f"     Mes(es) de interrupción: {row['Mes_Interrupcion_PDF']}")
            print(f"     Monto(s) compensado(s): {row['MONTO_PDF']}")
            print(f"     Cantidad: {row['Cantidad_Compensaciones']} compensación(es)")
            print()
    
    # Mostrar suministros sin PDF encontrado
    sin_pdf = df[df['Mes_Interrupcion_PDF'] == "PDF NO ENCONTRADO"]
    if len(sin_pdf) > 0:
        print(f"\n⚠️  SUMINISTROS SIN PDF ENCONTRADO ({len(sin_pdf)}):")
        for suministro in sin_pdf['Código de Suministro'].head(10):
            print(f"   - {suministro}")
        if len(sin_pdf) > 10:
            print(f"   ... y {len(sin_pdf) - 10} más")

def test_with_single_pdf():
    """
    Función para probar con un PDF específico
    """
    pdf_folder = r"C:\Users\P760033411\OneDrive - Pluz Energía Perú S.A.A\EJECUCION--PYTHON\Recibos 1\recibos"
    
    # Listar PDFs disponibles
    pdfs = glob(os.path.join(pdf_folder, "*.pdf")) + glob(os.path.join(pdf_folder, "*.PDF"))
    
    if not pdfs:
        print("❌ No se encontraron PDFs en la carpeta")
        return
    
    print(f"📁 PDFs encontrados: {len(pdfs)}")
    
    # Probar con el primer PDF
    pdf_path = pdfs[0]
    print(f"\n🔍 Probando extracción con: {os.path.basename(pdf_path)}")
    
    # Hacer análisis completo
    process_all_ds026_in_pdf(pdf_path)
    
    # Extraer compensaciones
    compensaciones = extract_ds026_from_pdf(pdf_path)
    
    print(f"\n{'='*60}")
    print("📊 RESULTADO DE LA PRUEBA:")
    if compensaciones:
        for i, (mes, monto) in enumerate(compensaciones, 1):
            print(f"  {i}. Mes de interrupción: {mes}, Monto compensado: {monto:.2f}")
    else:
        print("  No se encontraron compensaciones DS 026")

if __name__ == "__main__":
    print("🤖 SISTEMA DE EXTRACCIÓN DS 026 - VERSIÓN MEJORADA")
    print("=" * 60)
    
    # Primero probar con un PDF para verificar la extracción
    print("\n1️⃣ PRIMERO: Probando extracción con un PDF de muestra...")
    test_with_single_pdf()
    
    print("\n" + "=" * 60)
    print("2️⃣ AHORA: Procesando todos los registros del Excel...")
    print("=" * 60)
    
    main()

🤖 SISTEMA DE EXTRACCIÓN DS 026 - VERSIÓN MEJORADA

1️⃣ PRIMERO: Probando extracción con un PDF de muestra...
📁 PDFs encontrados: 298

🔍 Probando extracción con: RECIBOS_2025_09_1555029.pdf

 🔎 ANÁLISIS COMPLETO DEL PDF
 Archivo: RECIBOS_2025_09_1555029.pdf

 📄 PÁGINA 1:
   Línea 36: Tipo de Conexión C1.1 DS 026 Cal_RCarga 2025-06 -0.08
     ✅ Coincidencia encontrada: Mes=2025-06, Monto=-0.08

 📄 PÁGINA 2:

 📖 Extrayendo datos DS 026 del PDF: RECIBOS_2025_09_1555029.pdf
 📄 Encontrado en página 1, línea 36: Tipo de Conexión C1.1 DS 026 Cal_RCarga 2025-06 -0.08
 ✅ Extraído: Mes=202506, Monto=0.08
 📊 Total de compensaciones DS 026 encontradas: 1

📊 RESULTADO DE LA PRUEBA:
  1. Mes de interrupción: 202506, Monto compensado: 0.08

2️⃣ AHORA: Procesando todos los registros del Excel...
📖 Leyendo archivo Excel...
 ✅ Excel cargado. Filas: 149, Columnas: ['Ítem', 'Código de Suministro', 'Mes en que se hace efectiva la compensación', 'Monto compensado en soles', 'Tipo de cambio', 'Periodo de Cont